__Dataset__: [Link](https://www.kaggle.com/datasets/uciml/sms-spam-collection-dataset)

In [92]:
import numpy as np
import pandas as pd
# from typing import overload
import re
import string
from utils.constants import chat_words
import os
import ast

import nltk
NLTK_DIR = os.path.abspath("./env/nltk_data")
# Tokenizer
nltk.download('punkt', download_dir=NLTK_DIR, raise_on_error=True, quiet=True)
nltk.download('punkt_tab', download_dir=NLTK_DIR, raise_on_error=True, quiet=True)
# Correct spelling
nltk.download('words', download_dir=NLTK_DIR, raise_on_error=True, quiet=True)
# stopword removal
nltk.download('stopwords', download_dir=NLTK_DIR, raise_on_error=True, quiet=True)
# download wordnet
nltk.download('wordnet', download_dir=NLTK_DIR, raise_on_error=True, quiet=True)

nltk.data.path.insert(0, NLTK_DIR)

from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.metrics.distance import jaccard_distance
from nltk.util import ngrams
from nltk.corpus import words, stopwords
from nltk.stem.porter import PorterStemmer
from nltk.stem import WordNetLemmatizer

from collections import Counter

from sklearn.preprocessing import OneHotEncoder, LabelEncoder

import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import (pack_padded_sequence, pad_packed_sequence)

#### Preprocessing

In [3]:
DATAPATH = 'data/spam.csv'

dataset_v1 = pd.read_csv(
    DATAPATH,
    header=0,
    usecols=[0, 1],
    names=["label", "message"],
    encoding="utf-8",
    encoding_errors="ignore"
)

df = dataset_v1.copy()
df = df.dropna(subset=["label", "message"]).copy()
df["label"] = df["label"].astype(str).str.strip().str.lower()
df["message"] = df["message"].astype(str)

print(f"Messages: {len(df):,}")
print("Label counts:", df["label"].value_counts().to_dict())
df.sample(5)


Messages: 5,572
Label counts: {'ham': 4825, 'spam': 747}


,label,message
4292,ham,You best watch what you say cause I get drunk ...
3364,ham,"Sorry, I'll call later"
3730,ham,Isn't frnd a necesity in life? imagine urself ...
1150,ham,(That said can you text him one more time?)
2365,ham,Ok try to do week end course in coimbatore.


In [4]:
def remove_number_middle_of_text(text):
    pattern = re.compile(r'[a-zA-Z]+')
    text = pattern.sub(r"", text)
    return " ".join(text)

def remove_htmltags(text):
    pattern = re.compile(r"<.*?>")
    return pattern.sub(r"", text)

def remove_urls(text):
    pattern = re.compile(r"https?:\/\/\S+|www\.\S+")
    return pattern.sub(r"", text)

def remove_punctuations(text):
    excluded_punctuations = string.punctuation
    return text.translate(str.maketrans(" ", " ", excluded_punctuations))

def eloborate_chat_conversion(texts):
    new_text = []
    token_words = texts.split()
    
    for word in token_words:
        key = word.upper()
        if key in chat_words:
            new_text.append(chat_words[key])
        else:
            new_text.append(word)
    return " ".join(new_text)


def corrected_spelling(texts):
    results = []
    correct_words = [w.lower() for w in words.words()]
    correct_set = set(correct_words)
    text_tokens = texts.split()
    
    for word in text_tokens:
        lower_word = word.lower()
        # keep the correctly spelled words
        if lower_word in correct_set or not lower_word.isalpha() or len(lower_word) < 2:
            results.append(lower_word)
            continue
        
        incorrect_bigrams = set(ngrams(lower_word, 2))
        best_score, best_match = 1.0, word
        
        for w in correct_words:
            if w[0] == lower_word[0]:
                correct_bigrams = set(ngrams(w, 2))
                score = jaccard_distance(incorrect_bigrams, correct_bigrams)
                if score < best_score:
                    best_score, best_match = score, w
        results.append(best_match)
    return " ".join(results)

def remove_stopwords(texts):
    cleaned = []
    text_tokens = texts.split()
    stop_words = set(stopwords.words('english'))
    for word in text_tokens:
        # word is already lowered
        if word and word not in stop_words:
            cleaned.append(word)
    return " ".join(cleaned)


def remove_emoji(texts):
    text_tokens = texts.split()
    text = " ".join(text_tokens)
    emoji_pattern = re.compile(
        "["
        "\U0001F600-\U0001F64F"  # emoticons
        "\U0001F300-\U0001F5FF"  # symbols & pictographs
        "\U0001F680-\U0001F6FF"  # transport & map symbols
        "\U0001F1E0-\U0001F1FF"  # flags
        "\U00002700-\U000027BF"  # dingbats
        "\U000024C2-\U0001F251"
        "]+",
        flags=re.UNICODE
    )
    texts = emoji_pattern.sub(r'', text)
    return texts


def stemming_words(text_tokens):
    ps = PorterStemmer()
    stemmed_words = []
    return [stemmed_words.append(ps.stem(word)) for word in text_tokens]

def lemmatize_words(text_tokens):
    lemmatized_words = []
    wordnet_lemmatizer = WordNetLemmatizer()
    for word in text_tokens:
        lemmatized_words.append(wordnet_lemmatizer.lemmatize(word, pos='v'))
    return lemmatized_words


def text_preprocessing(text):
    # lowercasing
    text = text.lower()
        
    # remove html tags
    text = remove_htmltags(text)
    
    # remove urls from text
    text = remove_urls(text)
    
    text = remove_punctuations(text)
    
    text = eloborate_chat_conversion(text)
    
    text = corrected_spelling(text)
    
    text = remove_stopwords(text)
    
    text = remove_emoji(text)
    
    tokenized_text = word_tokenize(text)
    
    # tokenized_text = stemming_words(tokenized_text)
    
    tokenized_text = lemmatize_words(tokenized_text)
    
    
    return tokenized_text

In [ ]:
# # text_preprocessing("""helo amzing ADIH <p>hello world</p> <br/> https://helo_texts.com
# # https://www.helo_texts.com
# # http://localhost:5000"""+df['message'][0])
# df['tokens'] = df['message'].apply(text_preprocessing)
# corpus_tokens = df['tokens'].tolist()

#### Remove the empty tokens

In [ ]:
# null_indices = df[df['tokens'].apply(lambda x: len(x) == 0)].index
# df.drop(null_indices, inplace=True)

In [ ]:
# ' '.join(df['tokens'].iloc[0])

'go juror point crazy available bugi n great world la e buffet cine get amores wat'

#### For deleting the memory using gc

In [6]:
# import gc

# # del X_train
# gc.collect()

In [ ]:
# df['cleaned_text'] = df['tokens'].apply(' '.join)
# DATA_DIR = os.path.abspath("./data")
# df.to_csv(f"{DATA_DIR}/processed.csv", index=True)

#### Processed data

Need to fix the cleaned text

In [61]:
df = pd.read_csv(os.path.abspath("./data/processed_v1.csv"))
df.sample(5)


,Unnamed: 0,label,message,tokens,cleaned_text
1045,1048,ham,I walked an hour 2 c u! doesnt that show I car...,"['walker', 'hour', '2', 'c', 'doesnt', 'show',...",walker hour 2 c doesnt show care wont believe ...
137,137,ham,No no. I will check all rooms befor activities,"['check', 'room', 'activity']",check room activity
2423,2427,ham,Do you think i can move &lt;#&gt; in a week,"['think', 'move', 'lilt', 'week']",think move lilt week
1033,1036,ham,"No my blankets are sufficient, thx","['blanket', 'sufficient', 'thank']",blanket sufficient thank
3641,3647,ham,As per your request 'Maangalyam (Alaipayuthe)'...,"['per', 'request', 'mangal', 'ampalaya', 'hash...",per request mangal ampalaya hash set caller ca...


### Features and targets

In [93]:
def string_list_to_list(string_list: str):
    return ast.literal_eval(string_list)


X_text = df['cleaned_text'].fillna("").astype(str)
X_token = df['tokens'].apply(string_list_to_list)

y = df['label']

In [99]:
y.value_counts()

label
ham     4815
spam     747
Name: count, dtype: int64

#### Train, validation and Test data

In [171]:
# with `cleaned_text` columns
# X_train, X_temp, y_train, y_temp = train_test_split(X_text, y, test_size=0.3, random_state=42, stratify=y, shuffle=True)
# X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp, shuffle=True)

X_train, X_temp, y_train, y_temp = train_test_split(X_token, y, test_size=0.3, random_state=42, stratify=y, shuffle=True)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp, shuffle=True)

print(f'X_train: {X_train.shape}')
print(f'y_train: {y_train.shape}')
print('\n')
print(f'X_val: {X_val.shape}')
print(f'y_val: {y_val.shape}')
print('\n')
print(f'X_test: {X_test.shape}')
print(f'y_test: {y_test.shape}')

X_train: (3893,)
y_train: (3893,)


X_val: (834,)
y_val: (834,)


X_test: (835,)
y_test: (835,)


In [180]:
X_train.tolist()[:5][0]

['happy', 'birthday', 'youdith', 'lot', 'lover', 'nonrigid']

In [127]:
le = LabelEncoder()
y_train_num = le.fit_transform(y_train)
y_val_num = le.transform(y_val)
y_test_num = le.transform(y_test)

In [128]:
y_train.value_counts()

label
ham     3370
spam     523
Name: count, dtype: int64

#### Convert target to numerical

## Train

#### Build vocabulary

In [188]:
PAD_TOKEN = "<pad>"
UNK_TOKEN = "<unk>"

PAD_IDX = 0
UNK_IDX = 1

vocab = {
    PAD_TOKEN: PAD_IDX,
    UNK_TOKEN: UNK_IDX
}


# when using dataframes `cleaned_text` columns
# for sentence in X_train.fillna("").tolist():
#     for token in str(sentence).split():
#         if token not in vocab:
#             vocab[token] = len(vocab)


# when using dataframes `tokens` columns
for token_sentence in X_train.tolist():
    for token in token_sentence:
        if token not in vocab:
            vocab[token] = len(vocab)


In [189]:
print("Vocabulary size:", len(vocab))

Vocabulary size: 5845


In [194]:
# when using dataframes `cleaned_text` columns
# def text_to_indices(sentence, vocab):
#     numerical_sentences = []
#     for token in str(sentence).split():
#         token_id = vocab.get(token, UNK_IDX)
#         numerical_sentences.append(token_id)
#     return numerical_sentences


# def convert_to_numerical_sentence(texts: list|str):
#     # print(f"convert_to_numerical_sentence:texts {texts}")
#     X_train_numerical_sentences = []

#     if isinstance(texts, str):
#         texts_indices = text_to_indices(texts, vocab)
#         return texts_indices
#     else:
#         for sentence in texts:
#             texts_indices = text_to_indices(sentence, vocab)
#             X_train_numerical_sentences.append(texts_indices)
#         return X_train_numerical_sentences



# when using dataframes `tokens` columns
def text_to_indices(sentence, vocab):
    numerical_sentences = []
    for token in sentence:
        token_id = vocab.get(token, UNK_IDX)
        numerical_sentences.append(token_id)
    return numerical_sentences


def convert_to_numerical_sentence(texts: list|str):
    # print(f"convert_to_numerical_sentence:texts {texts}")
    X_train_numerical_sentences = []

    if isinstance(texts, str):
        texts_indices = text_to_indices(texts, vocab)
        return texts_indices
    else:
        for sentence in texts:
            texts_indices = text_to_indices(sentence, vocab)
            X_train_numerical_sentences.append(texts_indices)
        return X_train_numerical_sentences

In [195]:
X_train_numerical_sentences = convert_to_numerical_sentence(X_train.tolist())
X_train_numerical_sentences[:3]

[[2, 3, 4, 5, 6, 7], [8, 9, 10, 11, 12, 13, 14, 10], [15, 16, 17, 18, 19]]

#### Padding

In [196]:
TRAIN_SEQ_MAX_LEN = max([ len(seq) for seq in X_train_numerical_sentences ])

def padded_numerical_sentences(numeric_sentences: list[list] | list):
    # Empty input: same behavior as before (returns an empty list)
    if len(numeric_sentences) == 0:
        return []

    # 1D input like [1, 2, 3] -> treat as one sequence
    is_1d = not isinstance(numeric_sentences[0], (list, tuple))
    sequences = [numeric_sentences] if is_1d else numeric_sentences
    print(f'padded_numerical_sentences__:({sequences})')
    padded_seq = []
    for seq in sequences:
        seq = list(seq)[:TRAIN_SEQ_MAX_LEN]
        if len(seq) == 0:
            seq = [UNK_IDX]

        pad_seq = seq + ([PAD_IDX] * (TRAIN_SEQ_MAX_LEN - len(seq)))
        padded_seq.append(pad_seq)

    # 1D in -> 1D out, 2D in -> 2D out
    return padded_seq[0] if is_1d else padded_seq
    

X_train_padded_seqs = padded_numerical_sentences(X_train_numerical_sentences)

padded_numerical_sentences__:([[2, 3, 4, 5, 6, 7], [8, 9, 10, 11, 12, 13, 14, 10], [15, 16, 17, 18, 19], [20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 13, 35, 36, 37], [37, 38, 39, 40, 41], [42, 43, 44, 45, 46, 47, 48, 12, 17, 49, 50], [51, 52, 53, 54, 55, 52, 56, 57, 26, 58], [59, 60, 61, 62, 63, 64, 65], [66, 10, 67, 68], [69, 70, 71, 72, 73, 8, 74, 75, 76], [77, 5, 78, 79, 80, 78, 81], [82, 83, 11], [84, 24, 85], [86, 87, 88, 89, 88, 90, 88, 91, 88, 88, 5, 92, 93, 88, 94, 95, 96], [97, 98, 99, 100, 101, 102, 103, 87, 104, 94, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115], [116, 117, 8, 118, 119, 120, 121, 122], [123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 17, 133, 134, 135, 136, 137, 138], [66, 139, 140], [141, 8, 142, 143], [144, 17], [145, 49, 146], [147, 148, 149, 8], [150, 151, 152, 153, 83, 139, 154, 155, 156, 157, 158, 13], [49, 159, 160, 12, 161, 162, 163, 17, 164, 165], [17, 20, 13, 166, 66, 37, 167, 152, 66], [168, 169, 170], [171, 172, 173, 17

In [197]:
len(X_train_padded_seqs)

3893

### Preserving the actual sequence length

In [198]:
X_train_lengths = []
for seq in X_train_numerical_sentences:
    X_train_lengths.append(min(len(seq), TRAIN_SEQ_MAX_LEN))

# handle empty sequences
X_train_lengths = [max(length, 1) for length in X_train_lengths]

In [199]:
len(X_train_padded_seqs)

3893

#### OneHotEncoder

In [ ]:
# all_train_tokens = list({
#     token 
#     for line in X_train.tolist() 
#     if isinstance(line, str) 
#     for token in line.split() 
#     if token and len(token) > 0 and token not in (0, 0.0, "0", "0.0")
# })

In [ ]:
# len(all_train_tokens)
# all_train_tokens_arr = np.array(all_train_tokens).reshape(-1, 1)

In [ ]:
# ohe = OneHotEncoder(sparse_output=True, handle_unknown='ignore', dtype=np.int8)
# ohe.fit(all_train_tokens_arr)

# vocab_size = len(ohe.categories_[0])

# def doc_to_ohe(corpus, ohe_encoder=ohe):
#     ohe_corpus = []
#     for line in corpus.tolist():
#         # 1. Ensure line is a string to avoid AttributeError on floats/NaNs
#         if isinstance(line, str):
#             line_tokens = [
#                 tok for tok in line.split() 
#                 if tok and tok not in (0, 0.0, "0", "0.0")
#             ]
            
#             # 2. Only transform if there are valid tokens left
#             if line_tokens:
#                 ohe_line = ohe_encoder.transform(np.array(line_tokens).reshape(-1, 1)).toarray()
#                 ohe_corpus.append(ohe_line)
                
#     return ohe_corpus


In [ ]:
# X_train_encoded = doc_to_ohe(X_train)

# encoded_max_length = max(len(line) for line in X_train_encoded)
# print(encoded_max_length)


In [ ]:
# X_train_ohe = []

# for ohe_line in X_train_encoded:
#     # Pad each line to the maximum length with zeros
#     padded_line = np.pad(ohe_line, ((0, encoded_max_length - len(ohe_line)), (0, 0)), mode='constant')
#     X_train_ohe.append(padded_line)

In [ ]:
# X_train_ohe[0]

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(79, 6233), dtype=int8)

#### Bag Of Words

In [ ]:
# train_corpus = X_train
# test_corpus = X_test

# cv = CountVectorizer()
# cv_ngram = CountVectorizer(ngram_range=(1, 2))
# X_train_bow = cv.fit_transform(train_corpus)
# X_test_bow = cv.transform(test_corpus)

# X_train_bow_ngram = cv_ngram.fit_transform(train_corpus)
# X_test_bow_ngram = cv_ngram.transform(test_corpus)

In [ ]:
# print(f'After apply the BoW(Bag of Word) the shape: {X_train_bow.shape}')
# print(f'Vocabulary size: {len(cv.vocabulary_)}')
# cv.get_feature_names_out()[1000]

### TF-IDF

In [200]:
X_train[0]

['go',
 'juror',
 'point',
 'crazy',
 'available',
 'bugi',
 'n',
 'great',
 'world',
 'la',
 'e',
 'buffet',
 'cine',
 'get',
 'amores',
 'wat']

In [12]:
# tfidf = TfidfVectorizer()

# X_train_tfidf = tfidf.fit_transform(X_train).toarray()
# X_val_tfidf = tfidf.transform(X_val).toarray()
# X_test_tfidf = tfidf.transform(X_test).toarray()

In [13]:
# len(tfidf.vocabulary_)

5818

In [18]:
# X_train_tfidf[0]

array([0., 0., 0., ..., 0., 0., 0.], shape=(5818,))

#### Torch device

In [201]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f'System device: {device}')

System device: cuda


#### Dataset and Dataloader

In [139]:

# class SentimentDataset(Dataset):
#     def __init__(self, X, y):
#         self.X = torch.tensor(X, dtype=torch.float32)
#         self.y = torch.tensor(y, dtype=torch.long)
        
#     def __len__(self):
#         return len(self.X)
    
#     def __getitem__(self, idx):
#         return self.X[idx], self.y[idx]

class SentimentDataset(Dataset):
    def __init__(self, X, lengths, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.lengths = torch.tensor(lengths, dtype=torch.long)
        # For BCEWithLogitsLoss (binary classification)
        self.y = torch.tensor(y, dtype=torch.float32)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return (self.X[idx], self.lengths[idx], self.y[idx])



#### Prepare Sequence

In [202]:
X_train_numerical_sentences[:5]
# X_train_padded_seqs.shape

[[2, 3, 4, 5, 6, 7],
 [8, 9, 10, 11, 12, 13, 14, 10],
 [15, 16, 17, 18, 19],
 [20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 13, 35, 36, 37],
 [37, 38, 39, 40, 41]]

### Word2Vec

In [141]:
from gensim.models import KeyedVectors
import gensim.downloader as api
word2vec_model = api.load('word2vec-google-news-300')
# # wv_model = KeyedVectors.load_word2vec_format('GoogleNews-vectors-negative300.bin.gz', binary=True, limit=500000)

In [203]:
print(f'Word2Vec Vocabulary: {len(word2vec_model)}')
print(f'Vocabulary(text-to-index): {len(vocab)}')
print(f'Embedding dimension: {word2vec_model.vector_size}')

vector = word2vec_model['test']
print(f'Vector shape: {vector.shape}')

Word2Vec Vocabulary: 3000000
Vocabulary(text-to-index): 5845
Embedding dimension: 300
Vector shape: (300,)


#### Word2Vec embedding matrix

In [205]:
embedding_dim = 300
# vocab
vocab_size = len(vocab)
embedding_matrix = np.zeros(shape=(vocab_size, embedding_dim), dtype=np.float32)

found_words = 0
missing_words = 0


for word, idx in vocab.items():
    if word == PAD_TOKEN:
        continue
    vector = None
    if word in word2vec_model:
        vector = word2vec_model[word]
    # elif word.upper() in word2vec_model:
    #     vector = word2vec_model[word.upper()]
    
    
    if vector is not None:
        embedding_matrix[idx] = vector
        found_words += 1
    else:
        embedding_matrix[idx] = np.random.normal(0, 0.1, embedding_dim)
        missing_words += 1


print(f'Embedding matrix: {embedding_matrix.shape}')
print(f'Found words: {found_words}')
print(f'Missing words: {missing_words}')

Embedding matrix: (5845, 300)
Found words: 3983
Missing words: 1861


In [214]:
len(X_train_padded_seqs)

print(f'Shape of X_train_padded_seqs: (3893 x 79)')

Shape of X_train_padded_seqs: (3893 x 79)


In [218]:

# train_dataset = SentimentDataset(X_train_padded_seqs, y_train_num)
# train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)

train_dataset = SentimentDataset(X_train_padded_seqs, X_train_lengths, y_train_num)
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)

embedding_weights = torch.tensor(embedding_matrix, dtype=torch.float32)

In [224]:
for num_seq, lengths, labels in train_dataloader:
    print(f'{num_seq.shape}, {lengths.shape}, {labels.shape}')
    break

torch.Size([32, 79]), torch.Size([32]), torch.Size([32])


## Model

In [225]:

class SentimentNet(nn.Module):
    def __init__(self, vocab_size, embedding_dim=300, hidden_dim=128, dense_dim=128, dropout=0.3):
        super(SentimentNet, self).__init__()
        
        # 1. embeddings
        # self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim, padding_idx=0)
        self.embedding = nn.Embedding.from_pretrained(embedding_weights, freeze=False, padding_idx=0)
        
        # 2. Bidirectional LSTM
        self.lstm = nn.LSTM(input_size=embedding_dim, hidden_size=hidden_dim, batch_first=True, bidirectional=True)
        
        # BiLSTM produces hidden_dim * 2 features
        lstm_output_dim = hidden_dim * 2    
        
        # 3. Dropout
        self.dropout1 = nn.Dropout(dropout)
        
        # 4. Dense layer
        self.fc1 = nn.Linear(in_features=lstm_output_dim, out_features=dense_dim)
        self.relu = nn.ReLU()
        
        # 5. Additional dropout
        self.dropout2 = nn.Dropout(dropout)
        
        # 6. Output layer
        self.fc2 = nn.Linear(in_features=dense_dim, out_features=1)
        

    def forward(self, X, lengths):
        lengths_cpu = lengths.cpu()
        lengths_dev = lengths.to(X.device)
        # input - [batch, sequence]
        X = self.embedding(X)
        
        # output - [batch, sequence, embedding_dim]
        # Pack sequences to exclude padding from LSTM processing
        packed = pack_padded_sequence(X, lengths=lengths_cpu, batch_first=True, enforce_sorted=False)
        packed_output, _ = self.lstm(packed)
        
        lstm_output, _ = pad_packed_sequence(packed_output, batch_first=True, total_length=X.size(1))
        
        # output - [batch, sequence, hidden_dim * 2]
        
        # createe a mask for actual tokens
        mask = (
            torch.arange(X.size(1), device=X.device).unsqueeze(0) < lengths_dev.unsqueeze(1)
        )
        mask = mask.unsqueeze(-1)
        
        # masked mean pooling
        masked_output = lstm_output * mask
        
        # pooled = masked_output.sum(dim=1) / lengths.to(device).unsqueeze(1)
        pooled = masked_output.sum(dim=1) / lengths.to(device).unsqueeze(1).clamp_min(1)
        
        # output: [batch, hidden_dim * 2]
        
        # Dense transformation
        X = self.dropout1(pooled)
        
        X = self.fc1(X)
        X = self.relu(X)
        X = self.dropout2(X)
        
        # output logits: [batch, 1]
        logits = self.fc2(X)
        
        return logits.squeeze(-1)


In [227]:
vocab_size

5845

In [228]:
model = SentimentNet(
    vocab_size=vocab_size,
    embedding_dim=300,
    hidden_dim=128,
    dense_dim=128,
    dropout=0.3).to(device)

criterion = nn.BCEWithLogitsLoss()

optimizer = optim.Adam(params=model.parameters(), lr=0.001)

In [231]:
model.embedding.weight.shape

torch.Size([5845, 300])

### Training Loop

In [232]:

epochs = 10

for epoch in range(epochs):
    
    model.train()
    total_loss = 0.0
    
    for token_ids, lengths, labels in train_dataloader:
        token_ids = token_ids.to(device)
        labels = labels.float().to(device)
        
        optimizer.zero_grad()
        
        # forward pass
        logits = model(token_ids, lengths)
        
        # calculate loss
        loss = criterion(logits, labels)
        
        # backpropagation
        loss.backward()
        
        # update parameters
        optimizer.step()
        total_loss += loss.item()
    
    print(f'Epoch: {epoch+1}, Loss: {total_loss / len(train_dataloader):.4f}')




Epoch: 1, Loss: 0.1986
Epoch: 2, Loss: 0.0363
Epoch: 3, Loss: 0.0142
Epoch: 4, Loss: 0.0053
Epoch: 5, Loss: 0.0005
Epoch: 6, Loss: 0.0002
Epoch: 7, Loss: 0.0001
Epoch: 8, Loss: 0.0001
Epoch: 9, Loss: 0.0001
Epoch: 10, Loss: 0.0000


#### Test with validation

In [ ]:

model.eval()
with torch.no_grad():
    logits = model(token_ids)
